## Tarea 1 Unidad 1 - Introducción a la IA: Machine Learning Procesamiento de datos
### 1. Procesamiento avanzado de datos
Alan Josue Solano Perez

Tomando como ejemplo el archivo "escalar_o_no_escalar.ipynb" en donde se aplican transformaciones de columnas de tipo escala. 

Usar las clases de StandarScaler() y MaxMiScaler(). Subir en este apartado el código para el tratamiento de datos del dataset titanic(train.csv)

Pasos: 

1.- Separar los datos en train, evaluacion y prueba en un 70%, 15% y 15% respectivamente

2.- Escalar los datos, las columnas que se deben escalar son las de "age", "fare" y "SibSp". 

3.-  Evaluar los datos con el modelo arboles de clasificación y Knn 

4.- Guardar y subir el archivo .ipynb

In [17]:
#Importacion de librerias
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#### 1.- Separar los datos en train, evaluacion y prueba en un 70%, 15% y 15% respectivamente

In [19]:
# 1. Cargar y preparar datos
df = pd.read_csv('../data/titanic/train_titanic.csv', index_col="PassengerId")
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']
#mostramos el dataset arreglado
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S


In [9]:
# Separación: 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

#### 2.- Escalar los datos, las columnas que se deben escalar son las de "age", "fare" y "SibSp". 

In [22]:
# 2. Escalamiento de columnas específicas
cols_to_scale = ['Age', 'Fare', 'SibSp']

# Función para evaluar ambos modelos de un solo golpe
def evaluar_modelos(X_entrena, X_valida, y_entrena, y_valida, titulo):
    # 1. Árbol de Decisión
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_entrena, y_entrena)
    acc_dt = accuracy_score(y_valida, dt.predict(X_valida))
    
    # 2. kNN
    knn = KNeighborsClassifier()
    knn.fit(X_entrena, y_entrena)
    acc_knn = accuracy_score(y_valida, knn.predict(X_valida))
    
    return {'Escalamiento': titulo, 'DT_Accuracy': acc_dt, 'kNN_Accuracy': acc_knn}

#### 3.- Evaluar los datos con el modelo arboles de clasificación y Knn 

In [23]:
# 3. Evaluación (kNN como ejemplo sensible al escala)
# Creamos un arreglo para almacenar los resultados
resultados = []

# Sin Escalar
resultados.append(evaluar_modelos(X_train, X_val, y_train, y_val, 'Ninguno'))

# Con StandardScaler
X_tr_ss, X_va_ss = X_train.copy(), X_val.copy()
ss = StandardScaler()
X_tr_ss[cols_to_scale] = ss.fit_transform(X_train[cols_to_scale])
X_va_ss[cols_to_scale] = ss.transform(X_val[cols_to_scale])
resultados.append(evaluar_modelos(X_tr_ss, X_va_ss, y_train, y_val, 'StandardScaler'))

# Con MinMaxScaler
X_tr_mm, X_va_mm = X_train.copy(), X_val.copy()
mm = MinMaxScaler()
X_tr_mm[cols_to_scale] = mm.fit_transform(X_train[cols_to_scale])
X_va_mm[cols_to_scale] = mm.transform(X_val[cols_to_scale])
resultados.append(evaluar_modelos(X_tr_mm, X_va_mm, y_train, y_val, 'MinMaxScaler'))

# Crear tabla comparativa
df_res = pd.DataFrame(resultados)
print(df_res)

     Escalamiento  DT_Accuracy  kNN_Accuracy
0         Ninguno     0.776119       0.69403
1  StandardScaler     0.783582       0.80597
2    MinMaxScaler     0.776119       0.80597
